# Enrichment analysis

In [1]:

import GSEA_suit as gsea
import pandas as pd
import numpy as np
import os
import getting_threshold_genes as gtg


In [2]:
def get_selected_genes(csv_file):
    rank_genes = gtg.get_gene_rank(csv_file)
    alpha = gtg.get_inflexion_point_from_gene_rank(rank_genes)
    alpha = 0
    print(alpha)
    gene_rank_df = rank_genes[rank_genes["abs_coef"]>alpha]
    return gene_rank_df

In [12]:
def get_gene_rank_format_cat_temporal(csv_file):
    rank_genes = gtg.get_gene_rank_format_cat_temporal(csv_file)
    gene_rank_df = rank_genes[rank_genes["abs_coef"]>0]
    return gene_rank_df
    

In [4]:

def get_selected_genes_format_RF(csv_file):
    rank_genes = gtg.get_gene_rank_format_RF(csv_file)
    alpha = gtg.get_inflexion_point_from_gene_rank(rank_genes)
    gene_rank_df = rank_genes[rank_genes["abs_coef"]>alpha]
    return gene_rank_df

In [64]:
algorithm = "smote_catBoost"
database = "Tabula_Sapiens"
database = "KEGG_2021_Human"
database = "SysMyo_Muscle_Gene_Sets"
database = "WikiPathway_2023_Human"
database = "GO_Molecular_Function_2023"

In [65]:
csv_path = f"Results/feature_selection/RNAseq/{algorithm}/"
file_list = os.listdir(csv_path)
file_list = [f for f in file_list if f.endswith('_Symbols.csv')]
file_list

['Status_untrained_feature_selection_Symbols.csv',
 'Experiment_GSE129643_feature_selection_Symbols.csv',
 'Experiment_GSE167186_feature_selection_Symbols.csv',
 'Experiment_GSE152558_feature_selection_Symbols.csv',
 'Sex_male_feature_selection_Symbols.csv',
 'Status_Sarcopenia_feature_selection_Symbols.csv',
 'Status_Healthy_feature_selection_Symbols.csv',
 'feature_importance_Symbols.csv',
 'Experiment_GSE60590_feature_selection_Symbols.csv',
 'Experiment_GSE157585_feature_selection_Symbols.csv',
 'Experiment_GSE164471_feature_selection_Symbols.csv',
 'Sex_female_feature_selection_Symbols.csv',
 'Status_trained_feature_selection_Symbols.csv']

In [66]:
csv_file = 'RNAseq_All_abundances_adjusted_feature_selection_Symbols.csv'
csv_file

'RNAseq_All_abundances_adjusted_feature_selection_Symbols.csv'

In [67]:
sections = []
tms_sections=[]
for csv_file in file_list:#[csv_file]:
    try:
        if "random_forest" in algorithm:
            rank_list = get_selected_genes_format_RF(csv_path + csv_file)
        elif "catBoost" in algorithm:
            rank_list = get_gene_rank_format_cat_temporal(csv_path + csv_file)
        else:
            rank_list = get_selected_genes(csv_path + csv_file)
        symbols = rank_list["Symbol"].tolist()
        #symbols = rank_list["symbol"].tolist()
        symbols = list(set(symbols))
        symbols=symbols[1:]
        coef = rank_list["coef"].tolist()
        #coef = rank_list["V1"].tolist()
        coef = list(set(coef))
        coef=coef[1:]
        dict_list = {"name": csv_file, "symbols": symbols, "coef": coef, "description": "RNAseq Ridge L2 0.1"}
        tms_sections.append(dict_list)
        sets_ranks = {}
        for gene, coef in zip(rank_list["Symbol"], rank_list["abs_coef"]):
            if gene in sets_ranks:
                sets_ranks[gene] += coef
            else:
                sets_ranks[gene] = coef
        sets_ranks = {k: v for k, v in sorted(sets_ranks.items(), key=lambda item: item[1], reverse=True)}
        # asing the value to the dictionary on the position they have
        for i, gene in enumerate(sets_ranks):
            sets_ranks[gene] = i+1
        folder_save = f"Results/feature_selection/RNAseq/{algorithm}/csv_file/"
        results = gsea.get_GSEA(rank_dict=sets_ranks, database= database, min_size=5, outdir=folder_save)
        enrichment = results.res2d
        h_enrichment = enrichment[enrichment['FDR q-val']<0.2]
        dict_list[database] = enrichment
        sections.append(dict_list)

    except Exception as e:
        print("Error with file: ", csv_file, "error ", e)
        #break

# save sections
sections_df = pd.DataFrame(sections)
sections_df.to_csv(f"Results/feature_selection/RNAseq/{algorithm}/enriched/{database}_selected_genes_all.csv", index=False)


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "Ensembl":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "Ensembl":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consis

Error with file:  feature_importance_Symbols.csv error  Length mismatch: Expected axis has 4 elements, new values have 3 elements


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "Ensembl":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "Ensembl":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consis

In [54]:
sections_df

,name,symbols,coef,description,SysMyo_Muscle_Gene_Sets
0,Status_untrained_feature_selection_Symbols.csv,"[IRX4, AIPL1, C17orf49, ARL8B, IZUMO2, GRWD1, ...","[1.970115291452536, 2.9042267150826917, 3.9094...",RNAseq Ridge L2 0.1,Name ...
1,Experiment_GSE129643_feature_selection_Symbols...,"[ACTR3P2, CAPN1, ELP4, ZNF121, MYH7B, LRPPRC, ...","[1.7582425318889152, 2.166507251782825, 2.6640...",RNAseq Ridge L2 0.1,Name ...
2,Experiment_GSE167186_feature_selection_Symbols...,"[PRRG1, AP2S1, RPS20P28, ACTR3P2, PTCHD1, TP53...","[0.5458501099658291, 2.057193005249359, 0.0739...",RNAseq Ridge L2 0.1,Name ...
3,Experiment_GSE152558_feature_selection_Symbols...,"[ATG9B, UBE2Q2, IQCH, LOC100418961, GNAZ, LOC1...","[1.9901279220570791, 2.8079211032501594, 3.758...",RNAseq Ridge L2 0.1,Name ...
4,Sex_male_feature_selection_Symbols.csv,"[GAL3ST3, ACTR3P2, MYBPH, ESR1, SFXN4, NLN, CF...","[1.9736713356246511, 2.762512760098545, 3.4881...",RNAseq Ridge L2 0.1,Name ...
5,Status_Sarcopenia_feature_selection_Symbols.csv,"[UBE2Q2, CTRL, TRAPPC12, HPS4, NTM, LHFPL2, CA...","[1.3091729359696629, 2.6890252686844542, 2.096...",RNAseq Ridge L2 0.1,Name ...
6,Status_Healthy_feature_selection_Symbols.csv,"[ND5, H2BC4, MDM2, ASZ1, PPP1CC, DNAH10, RPL13...","[1.7266723137657811, 2.764313822893312, 2.5539...",RNAseq Ridge L2 0.1,Name ...
7,Experiment_GSE60590_feature_selection_Symbols.csv,"[MYO1H, DNM1L, PA2G4, HPCAL4, TP53BP2, ICE2, D...","[0.1732370681046982, 2.4617436827937995, 3.220...",RNAseq Ridge L2 0.1,Name ...
8,Experiment_GSE157585_feature_selection_Symbols...,"[ACTR3P2, STAT5A, ZNF121, LRPPRC, PGAP4, CSNK1...","[1.875225423016472, 2.8811194546381182, 2.6983...",RNAseq Ridge L2 0.1,Name ...
9,Experiment_GSE164471_feature_selection_Symbols...,"[RNA5SP35, HMGN3, ABCB4, ARHGAP11B, GPD2, DAAM...","[0.5298044214412352, 2.374448191875903, 2.7162...",RNAseq Ridge L2 0.1,Name ...


In [43]:
tms_sections

[{'name': 'Status_untrained_feature_selection_Symbols.csv',
  'symbols': ['IRX4',
   'AIPL1',
   'C17orf49',
   'ARL8B',
   'IZUMO2',
   'GRWD1',
   'HPCAL4',
   'CCL18',
   'LOC100287355',
   'ECI1',
   'LOC100128865',
   'ZNF580',
   'EFEMP2',
   'THOC3',
   'TMEM232',
   'SLC35A4',
   'TOMM22',
   'ATP6V1A',
   'BCAM',
   'CCDC34',
   'TTTY14',
   'SMAP1',
   'GOLGA6L6',
   'RWDD4',
   'CFLAR',
   'KLHL22',
   'STIP1',
   'ST6GALNAC2',
   'PHBP4',
   'ZNF234',
   'PRPS1L1',
   'DNM1',
   'RPL7P44',
   'RTKN',
   'SMCO1',
   'SORBS1',
   'TACSTD2',
   'AP2A2',
   'VAPA',
   'SETD6',
   'CTTNBP2',
   'APBB2',
   'TRABD',
   'RASGRP1',
   'C7orf25',
   'ELP6',
   'TRPV2',
   'MTND2P9',
   'DNM3',
   'CTCF',
   'ELMO2',
   'PI4KA',
   'SPCS1',
   'ARHGAP1',
   'SMTNL2',
   'CLK1',
   'MYO1A',
   'RPL12P26',
   'UPK3BL2',
   'ZNF219',
   'TPRG1',
   'HIP1R',
   'DSN1',
   'C5orf38',
   'SLC5A6',
   'HSP90AB1',
   'TMEM164',
   'GGA3',
   'PPM1AP1',
   'NBN',
   'CPM',
   'EYA3',
   'TXLN

In [49]:
h_enrichment#["Term"]

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes


In [68]:
terms=[]
for enrichment in sections_df[database]:
    terms.append(set(enrichment["Term"].tolist()))
terms

[{'ATP Binding (GO:0005524)',
  'Actin Binding (GO:0003779)',
  'Adenyl Ribonucleotide Binding (GO:0032559)',
  'Cadherin Binding (GO:0045296)',
  'Calcium Ion Binding (GO:0005509)',
  'Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000987)',
  'DNA Binding (GO:0003677)',
  'GTPase Activity (GO:0003924)',
  'GTPase Binding (GO:0051020)',
  'GTPase Regulator Activity (GO:0030695)',
  'Kinase Binding (GO:0019900)',
  'Metal Ion Binding (GO:0046872)',
  'Microtubule Binding (GO:0008017)',
  'Protein Homodimerization Activity (GO:0042803)',
  'Protein Kinase Binding (GO:0019901)',
  'Protein Serine/Threonine Kinase Activity (GO:0004674)',
  'Purine Ribonucleoside Triphosphate Binding (GO:0035639)',
  'RNA Binding (GO:0003723)',
  'RNA Polymerase II Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000978)',
  'RNA Polymerase II Transcription Regulatory Region Sequence-Specific DNA Binding (GO:0000977)',
  'Ribonucleoside Triphosphate Phosphatase Activity (GO:0017111)',
  '

In [69]:
intersection = set.intersection(*terms)
intersection

{'Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000987)',
 'DNA Binding (GO:0003677)',
 'GTPase Regulator Activity (GO:0030695)',
 'Protein Homodimerization Activity (GO:0042803)',
 'Purine Ribonucleoside Triphosphate Binding (GO:0035639)',
 'RNA Binding (GO:0003723)',
 'RNA Polymerase II Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000978)',
 'RNA Polymerase II Transcription Regulatory Region Sequence-Specific DNA Binding (GO:0000977)',
 'Sequence-Specific Double-Stranded DNA Binding (GO:1990837)'}

In [24]:
# name and terms
names = sections_df["name"].tolist()
name_terms = zip(names, terms)
name_terms_df = pd.DataFrame(name_terms, columns=["name", "terms"])

In [36]:
muscles = []
for row in name_terms_df.iterrows():
    term = row[1]["terms"]
    m = False
    for t in term:
        if "muscle" in t.lower() or "myo" in t.lower() or "sarcomere" in t.lower() or \
        "actin" in t.lower() or "myosin" in t.lower() or "fibroblast" in t.lower():
            m = True
            break

    muscles.append(m)
        
name_terms_df["muscle"] = muscles

In [37]:
name_terms_df

,name,terms,muscle,fibroblast
0,Experiment_GSE167186_feature_selection_Symbols...,"{Bone Marrow-erythrocyte, Eye-erythroid Lineag...",False,False
1,Sex_male_feature_selection_Symbols.csv,"{Salivary Gland-monocyte, Vasculature-smooth M...",True,False
2,Status_Sarcopenia_feature_selection_Symbols.csv,{Thymus-erythrocyte},False,False
3,Status_Healthy_feature_selection_Symbols.csv,"{Heart-cardiac Muscle Cell, Prostate-basal Cel...",True,False
4,Experiment_GSE60590_feature_selection_Symbols.csv,"{Bone Marrow-erythrocyte, Thymus-erythrocyte, ...",True,False
5,Experiment_GSE164471_feature_selection_Symbols...,"{Vasculature-pericyte Cell, Vasculature-smooth...",True,False
6,Sex_female_feature_selection_Symbols.csv,"{Muscle-fast Muscle Cell, Thymus-fast Muscle C...",True,False


In [ ]:
list(enrichment['Term'])

In [ ]:
sets_ranks

{'ATP1B4': 1,
 'HMGB2': 2,
 0: 3,
 'FEZ2': 4,
 'CSDE1': 5,
 'RNA5SP123': 6,
 'MTCO1P12': 7,
 'C12orf75': 8,
 'RNA5SP66': 9,
 'LOC440311': 10,
 'NDUFB4': 11,
 'NPY6R': 12,
 'TARS1': 13,
 'IRX3': 14,
 'MYLK4': 15,
 'CFAP61': 16,
 'ACHE': 17,
 'EPC1': 18,
 'RNF10': 19,
 'DUSP3': 20,
 'ANKRD12': 21,
 'MTRNR2L10': 22,
 'XPO5': 23,
 'ERCC1': 24,
 'TLN1': 25,
 'SLC37A4': 26,
 'RNA5SP191': 27,
 'ECH1': 28,
 'BCLAF3': 29,
 'WDR74': 30,
 'AQP1': 31,
 'TRAJ27': 32,
 'PSME4': 33,
 'ACTB': 34,
 'ARHGEF6': 35,
 'KIF2A': 36,
 'RNA5SP237': 37,
 'HK2': 38,
 'PCID2': 39,
 'CDIN1': 40,
 'S100PBP': 41,
 'FUCA2': 42,
 'ERCC5': 43,
 'SNURF': 44,
 'NUDCD3': 45,
 'MTRR': 46,
 'COPZ2': 47,
 'TFDP2': 48,
 'RNA5SP75': 49,
 'RFTN1': 50,
 'RRAS2': 51,
 'PDGFRB': 52,
 'RPS25': 53,
 'NUDT19': 54,
 'ZNF232': 55,
 'TRMT13': 56,
 'MLF1': 57,
 'IGFN1': 58,
 'HCN1': 59,
 'EIF2S1': 60,
 'MAB21L3': 61,
 'MCCC1': 62,
 'TLE1': 63,
 'MTRNR2L2': 64,
 'KIAA1143': 65,
 'POLR3D': 66,
 'CHRNB1': 67,
 'METTL25': 68,
 'MROH7': 69,
 

In [ ]:
lab_mirna_targets= ["gap43",
"acvr2a",
"hdac9",
"grp78",
"atf6",
"pgc1-a",
"acvr1b",
"NCAM",
"Hmbox1",
"p2ry6",
"camk2a",
"cadps",
"lch1",
"Il1rapl1",
"Carf",
"Aak1",
"P2ry1",
"Yy1",
"Psme3",
"Bach2",
"Smurf2",
"Atp6v1g1",
"Kcnk10",
"Gpx4",
"Psmg4",
"bach2",
"pten",
"p62",
"Prdx6"]
# upper case
lab_mirna_targets = [x.upper() for x in lab_mirna_targets]

In [ ]:
lab_mirna_targets

['GAP43',
 'ACVR2A',
 'HDAC9',
 'GRP78',
 'ATF6',
 'PGC1-A',
 'ACVR1B',
 'NCAM',
 'HMBOX1',
 'P2RY6',
 'CAMK2A',
 'CADPS',
 'LCH1',
 'IL1RAPL1',
 'CARF',
 'AAK1',
 'P2RY1',
 'YY1',
 'PSME3',
 'BACH2',
 'SMURF2',
 'ATP6V1G1',
 'KCNK10',
 'GPX4',
 'PSMG4',
 'BACH2',
 'PTEN',
 'P62',
 'PRDX6']

In [ ]:
sections_df

,name,symbols,coef,description,Tabula_Sapiens
0,feature_importance_Symbols.csv,"[USP36, RFK, CDK2AP1, CETN2, MICAL2, CHRNB1, T...","[0.9576554952824688, 0.4387035937214147, 0.040...",RNAseq Ridge L2 0.1,"Series([], Name: Term, dtype: object)"


In [ ]:
for row in sections_df.iterrows():
    symbols = row[1]["symbols"]
    known_genes = [gene for gene in symbols if gene in lab_mirna_targets]
    print (row[1]["name"], len(known_genes))

feature_importance_Symbols.csv 0


In [ ]:
symbols

['USP36',
 'RFK',
 'CDK2AP1',
 'CETN2',
 'MICAL2',
 'CHRNB1',
 'TASOR2',
 'RNA5SP258',
 'PPP2R1B',
 'LZIC',
 'PRKAG1',
 'TDG',
 'RBSN',
 'XIRP1',
 'MT1X',
 'COL1A2',
 'PITHD1',
 'MTRNR2L3',
 'MYLIP',
 'WDR74',
 'S100PBP',
 'MTRNR2L11',
 'QKI',
 'ND1',
 'IGHV1-24',
 'DUSP3',
 'EPC1',
 'RFTN1',
 'PCID2',
 'MTND4P24',
 'MCCC1',
 'RNA5SP75',
 'RPL5',
 'HACD1',
 'PPP1R1A',
 'TRMT13',
 'POLK',
 'PDGFRB',
 'IFITM1',
 'SNRNP27',
 'TMEM87A',
 'SAMD4A',
 'ROCK1',
 'MYLK4',
 'RNA5SP123',
 'ITFG2',
 'DDX10',
 'IMPA2',
 'SRP14P4',
 'RPL18A',
 'FTSJ1',
 'MTRR',
 'RRAS2',
 'NUCKS1',
 'CNBD2',
 'SLC37A4',
 'EPB41L3',
 'TRAJ21',
 'MBTD1',
 'TBX15',
 'MTRNR2L10',
 'HMGB2',
 'ACHE',
 'FAM98B',
 'PIK3R1',
 'MYH11',
 'MIER3',
 'C1QTNF6',
 'RNA5SP408',
 'NFKBIA',
 'FUCA2',
 'ERBB3',
 'FEZ2',
 'GPN3',
 'SLF2',
 'ARHGEF6',
 'CREB1',
 'MN1',
 'FBXO4',
 'TARS1',
 'RPRD1B',
 'FAM83A',
 'CFAP61',
 'RNA5SP445',
 'METTL25',
 'RNA5SP191',
 'ERCC5',
 'RPS25',
 'MTSS1',
 'HECTD2',
 'MTCO1P12',
 'DYNC1I2',
 'TLN1',
 'H